# 31 · Opto — post-shift phase

**Scope.** PPC silencing right after a distribution shift (Uniform → Hard-A/B),
when the statistical model is wrong and must be updated. The mirror of NB30: there
the circuit prediction was a **null** (model adequate → PPC dispensable); here PPC
should **matter**, and matter *more* than in the expert phase.

**The test has two parts.**
1. **Confirmation (§2)** — does silencing move behaviour post-shift *at all*? The
   gate the whole story rests on: the expert null (NB30) only reads as
   "dispensable" if the laser provably *can* move behaviour where it should. HET-Δ
   vs WT-Δ per phase (one Δ, 6 vs 4, floors ~p≈0.0095).
2. **Dispensability interaction (§3)** — does the opto effect *grow* from expert to
   post-shift? Per animal, Δ_post-shift − Δ_expert. The headline claim, but a
   difference-of-differences on the paired set only — **underpowered / directional**
   (within-HET Wilcoxon at n=6 floors at p=0.031).

**Hard-A and Hard-B stay separate.** Mirror distributions; the history stats
(recency, win/lose) are in principle mirror-invariant, but that has not been shown
here, so A and B are never pooled into a single 'hard' view. The lateralised curve
params (`pse`, lapse) flip sign across the mirror — per phase by necessity (§3b) —
and the update matrices (§5) keep A and B apart. So the interaction (§3) runs per
hard phase: uniform → Hard-A and uniform → Hard-B.

Unit of inference is the animal throughout (pool within animal → one value per
animal → test across animals). Genotype is injected — the loader doesn't carry it.


In [ ]:
from shared_setup import *
from analysis.phase import compute_group_phase
from analysis.opto import extract_opto_estimates
from behav_utils.analysis import paired_diff, rank_test
from plotting.opto import plot_delta_swarm, plot_delta_paired
from behav_utils.plotting import plot_um
import warnings
from IPython.display import display

experiment, info = load_data()

# ── config ───────────────────────────────────────────────────────────────────
# Genotype is NOT in the data — inject it. Keep in sync as the cohort grows.
# het = ChR2+ (inhibition, effect of interest); wt = transgene-negative control.
GENOTYPE = {'SS14': 'wt', 'SS17': 'wt', 'SS18': 'wt', 'SS20': 'wt',
            'SS15': 'het', 'SS16': 'het', 'SS19': 'het',
            'SS21': 'het', 'SS22': 'het', 'SS23': 'het'}
for aid, animal in experiment.animals.items():
    if aid in GENOTYPE:
        animal.metadata['genotype'] = GENOTYPE[aid]
WT_IDS   = [a for a in GENOTYPE if GENOTYPE[a] == 'wt'  and a in experiment.animals]
HET_IDS  = [a for a in GENOTYPE if GENOTYPE[a] == 'het' and a in experiment.animals]
OPTO_IDS = WT_IDS + HET_IDS

SYM_STATS    = ['recency', 'win_stay', 'lose_shift']        # symmetric — read per phase
LAT_STATS    = ['pse', 'slope', 'lapse_low', 'lapse_high']  # lateralised — per phase, lapses apart
LEAD         = 'recency'                                    # robust, symmetric — the headline stat
N_BOOT_CURVE = 200                                          # within-animal curve-param CI (points_lat only)
PHASE_LIST   = ['uniform', 'hard_a', 'hard_b']
PHASES       = {'uniform': 'Uniform', 'hard_a': 'Hard-A', 'hard_b': 'Hard-B'}

print(f"HET ({len(HET_IDS)}): {HET_IDS}")
print(f"WT  ({len(WT_IDS)}): {WT_IDS}")
if not HET_IDS or not WT_IDS:
    print("One genotype group is empty — check the GENOTYPE map against loaded animals.")

In [ ]:
def genotype_contrast(delta_df, stat, phase, a='het', b='wt'):
    """HET-Δ vs WT-Δ within one phase (Mann–Whitney). Returns (p, n_a, n_b)."""
    d = delta_df[(delta_df['stat'] == stat) & (delta_df['phase'] == phase)]
    va = d[d['genotype'] == a]['delta'].dropna().to_numpy()
    vb = d[d['genotype'] == b]['delta'].dropna().to_numpy()
    return rank_test(va, vb, paired=False)['p'], len(va), len(vb)


def interaction(delta_df, stat, pa, pb):
    """Δ_pb vs Δ_pa: within-HET paired Wilcoxon + HET-vs-WT on the difference-of-
    differences (paired_diff applied to the Δ table by phase). Paired set only."""
    sub = delta_df[delta_df['stat'] == stat][['animal', 'genotype', 'stat', 'phase', 'delta']]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')             # missing-phase drops are expected
        D = paired_diff(sub, by='phase', a=pb, b=pa, value='delta', keys=('animal',))
    if pa not in D.columns or pb not in D.columns or len(D) == 0:
        return dict(stat=stat, within_het_p=np.nan, het_vs_wt_p=np.nan,
                    n_paired_het=0, n_paired_wt=0)
    h, wt = D[D['genotype'] == 'het'], D[D['genotype'] == 'wt']
    within = rank_test(h[pb], h[pa], paired=True) if len(h) else {'p': np.nan}
    cross  = rank_test(h['delta'], wt['delta'], paired=False) if len(h) and len(wt) else {'p': np.nan}
    return dict(stat=stat, within_het_p=within['p'], het_vs_wt_p=cross['p'],
                n_paired_het=len(h), n_paired_wt=len(wt))

## §1 · Per-animal aggregation

`extract_opto_estimates` pools each animal's opto sessions (`pool_arrays`) and
computes the symmetric history stats, one value per (animal, phase, condition);
`paired_diff` reduces to Δ = opto − nonopto. Uniform / Hard-A / Hard-B are kept
**separate** — A ≡ B is not assumed, so they are never pooled.


In [ ]:
points_sym = extract_opto_estimates(experiment, phases=PHASE_LIST, stats=SYM_STATS,
                                    animals=OPTO_IDS, n_boot=0)
delta_sym  = (paired_diff(points_sym, by='trial_type', a='opto', b='opto_off')
              .rename(columns={'distribution': 'phase'}))

ORDER = [LEAD] + [s for s in SYM_STATS if s != LEAD]
print(f'Δ {LEAD} per animal across phases (opto − nonopto):')
display(delta_sym[delta_sym['stat'] == LEAD]
        .pivot_table(index=['animal', 'genotype'], columns='phase', values='delta')
        .round(3))

### §1b · Lateralised curve params  *(slow)*

Fits the psychometric per condition with a within-animal bootstrap CI
(`N_BOOT_CURVE`) across all three phases — the slow step. The history stats (§2–§3)
don't need it, so run those first; only §3b/§3c need `delta_lat`. Drop
`N_BOOT_CURVE` to iterate.


In [ ]:
points_lat = extract_opto_estimates(experiment, phases=PHASE_LIST, stats=['psychometric'],
                                    animals=OPTO_IDS, n_boot=N_BOOT_CURVE)
delta_lat  = (paired_diff(points_lat, by='trial_type', a='opto', b='opto_off')
              .rename(columns={'distribution': 'phase'}))
# psychometric expands to mu / sigma / lapse_low / lapse_high — rename to the curve
# names used here. lapses stay SEPARATE: not mirror-symmetric, so averaging hides it.
delta_lat['stat']  = delta_lat['stat'].replace({'mu': 'pse', 'sigma': 'slope'})
points_lat['stat'] = points_lat['stat'].replace({'mu': 'pse', 'sigma': 'slope'})
print('lateralised Δ ready for:', sorted(delta_lat['stat'].unique()))

## §2 · Confirmation: silencing moves behaviour?  *(gate)*

Per phase (uniform / Hard-A / Hard-B), HET-Δ vs WT-Δ. A flat null in **HET** too
would make the expert null uninterpretable — the laser may simply do nothing — so
this is the gate, and recency (robust) leads. A and B are shown separately; for the
symmetric stats they should look broadly alike, which is itself the mirror check.


In [ ]:
# per-phase confirmation: HET-Δ vs WT-Δ, recency leading
for phase in PHASE_LIST:
    conf = pd.DataFrame([dict(stat=st,
                              **dict(zip(['p_HET_vs_WT', 'n_het', 'n_wt'],
                                         genotype_contrast(delta_sym, st, phase))))
                         for st in ORDER])
    print(f'\n{PHASES[phase]}: Δ (opto − nonopto), HET vs WT')
    display(conf.round(4))

    fig, axes = plt.subplots(1, len(ORDER), figsize=(3.4 * len(ORDER), 3.4))
    for ax, st in zip(np.atleast_1d(axes), ORDER):
        p = conf.loc[conf['stat'] == st, 'p_HET_vs_WT'].iloc[0]
        plot_delta_swarm(delta_sym[delta_sym['phase'] == phase], st, ax=ax, p_value=p)
    fig.suptitle(f'{PHASES[phase]}: Δ (opto − nonopto), HET vs WT', y=1.03)
    fig.tight_layout(); plt.show()

## §3 · Dispensability interaction: Δ expert vs Δ post-shift  *(primary)*

Does the opto effect *grow* from expert to post-shift? Run **per hard phase**
(uniform → Hard-A, uniform → Hard-B). `within_het_p` is the paired Wilcoxon across
HET (Δ_hard vs Δ_uniform); `het_vs_wt_p` asks whether that growth is specific to
inhibition (HET) and not the light artefact (WT), via the difference-of-differences.
Both are underpowered and use only the paired set (`n_paired_*`) — read as
direction-of-effect, not proof. A consistent steepening in HET (not WT) is the
signature.


In [ ]:
# dispensability interaction, per hard phase
for hard_phase in ['hard_a', 'hard_b']:
    inter = pd.DataFrame([interaction(delta_sym, st, 'uniform', hard_phase) for st in ORDER])
    print(f'\nuniform → {PHASES[hard_phase]}: dispensability interaction')
    display(inter.round(4))

    fig, axes = plt.subplots(1, len(ORDER), figsize=(3.9 * len(ORDER), 3.8))
    for ax, st in zip(np.atleast_1d(axes), ORDER):
        p = inter.loc[inter['stat'] == st, 'within_het_p'].iloc[0]
        plot_delta_paired(delta_sym, st, ax=ax,
                          phase_a='uniform', phase_b=hard_phase, p_value=p)
    fig.suptitle(f'Dispensability: per-animal Δ expert → Δ {PHASES[hard_phase]}', y=1.03)
    fig.tight_layout(); plt.show()

## §3b · Lateralised curve params, per phase

`pse`, `slope` and the two lapses from the curve fit. `pse` and lapse are
lateralised — a pooled A+B view cancels them — so they're shown for uniform /
Hard-A / Hard-B separately. All rest on the per-condition curve fit, the thin and
less reliable readout post-shift (see §6); secondary to the history stats.


In [ ]:
for phase in PHASE_LIST:
    fig, axes = plt.subplots(1, len(LAT_STATS), figsize=(3.4 * len(LAT_STATS), 3.4))
    for ax, st in zip(np.atleast_1d(axes), LAT_STATS):
        p, _, _ = genotype_contrast(delta_lat, st, phase)
        plot_delta_swarm(delta_lat[delta_lat['phase'] == phase], st, ax=ax, p_value=p)
    fig.suptitle(f'{PHASES[phase]}: Δ curve params, HET vs WT  (per phase — not pooled)', y=1.03)
    fig.tight_layout(); plt.show()

## §3c · Curve-param dispensability  *(paired)*

The expert→post-shift paired view (as §3) for the curve params, per hard phase.
`slope` is symmetric, so its Hard-A and Hard-B panels should agree — a free check;
`pse` and the two lapses may legitimately differ between A and B. All rest on the
thin per-condition fit (§6) — directional at most.


In [ ]:
HARD = ['hard_a', 'hard_b']
lat_inter = pd.DataFrame([{'hard_phase': hp, **interaction(delta_lat, st, 'uniform', hp)}
                          for st in LAT_STATS for hp in HARD])
display(lat_inter.round(4))

fig, axes = plt.subplots(len(LAT_STATS), len(HARD),
                         figsize=(4.1 * len(HARD), 3.7 * len(LAT_STATS)), squeeze=False)
for i, st in enumerate(LAT_STATS):
    for j, hp in enumerate(HARD):
        p = lat_inter[(lat_inter['stat'] == st) &
                      (lat_inter['hard_phase'] == hp)]['within_het_p'].iloc[0]
        plot_delta_paired(delta_lat, st, ax=axes[i][j],
                          phase_a='uniform', phase_b=hp, p_value=p)
        axes[i][j].set_title(f"{st}: uniform → {PHASES[hp]}", fontsize=10)
fig.suptitle('Curve-param dispensability: Δ expert → Δ post-shift, per hard phase', y=1.005)
fig.tight_layout(); plt.show()

## §5 · Post-shift update matrices

Group-mean update matrices — nonopto / opto / difference — per genotype, Hard-A and
Hard-B kept separate (pooling mirror distributions would blur the structure).
Mega-pooled across the group, given how thin post-shift opto trials are. The
difference matrix carries both conditions' noise, so read it for gross structure
only — the SC→BE-reversion prediction lives here but needs more trials to test.


In [ ]:
for phase in ['hard_a', 'hard_b']:
    for label, ids in [('HET', HET_IDS), ('WT', WT_IDS)]:
        if not ids:
            continue
        _, um = compute_group_phase(experiment, phase, ids, method='pooled', n_bootstrap=0)
        u_off, u_on = um.get('opto_off'), um.get('opto_on')
        fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
        for ax, u, ttl in [(axes[0], u_off, 'nonopto'), (axes[1], u_on, 'opto')]:
            if u is not None:
                plot_um(u, ax=ax); ax.set_title(f'{label} {PHASES[phase]} — {ttl}', fontsize=9)
            else:
                ax.set_visible(False)
        if u_on is not None and u_off is not None:
            diff = np.asarray(u_on['um'], float) - np.asarray(u_off['um'], float)
            vmax = float(np.nanmax(np.abs(diff))) or 1.0
            plot_um(diff, ax=axes[2], vmin=-vmax, vmax=vmax)
            axes[2].set_title(f'{label} — Δ (opto − nonopto)', fontsize=9)
        else:
            axes[2].set_visible(False)
        fig.tight_layout(); plt.show()

## §6 · Data check: trial counts per phase × condition

How thin is post-shift? Trials per animal per condition (opto / nonopto / post), by
phase — the honest backdrop to the power caveats above. The §2–§3 tests are only as
strong as these counts allow.


In [ ]:
# trials per animal per phase × condition — the honest backdrop to power
qc = points_sym[points_sym['stat'] == LEAD]
display(qc.pivot_table(index=['genotype', 'animal'], columns=['distribution', 'trial_type'],
                       values='n_trials', fill_value=0).astype(int))